In [1]:
import random

# Functions
## Text parsing functions

In [2]:
def read_text_file(path):
    """
    Reads text file
    """
    with open(path, "r", encoding="utf-8") as f:
        new_text = f.read().lower()
        return new_text

In [3]:
def text_to_list(new_text, separator="\n\n", START="*S*", END="*E*", NL="*NL*"):
    """
    Converts text into a token sequence with START/END/NL markers.
    Args:
        new_text: New text.
        separator: String that splits the text into chunks. END, START are inserted between chunks.
            If None, the entire file is treated as a single chunk.
        START: Token to mark the start of a chunk.
        END: Token to mark the end of a chunk.
        NL: Placeholder for newlines.
    Returns:
        List of lowercase tokens including START, END and NL indicators.
    """
    
    if separator is None:
        chunks = [new_text]
    else:
        chunks = new_text.split(separator)

    text_list = [START]

    for i, chunk in enumerate(chunks):
        chunk = chunk.strip()
        if chunk:
            # explicitly preserve within-chunk newlines
            chunk = chunk.replace("\n", f" {NL} ")

            # split on any whitespace
            text_list.extend(chunk.split())

        if i != len(chunks) - 1:
            # if there are still more chunks, do END START
            text_list.append(END)
            text_list.append(START)

    text_list.append(END)
    return text_list

In [4]:
def list_to_text(text_list, separator="\n\n", avoid_double_newlines=True, START="*S*", END="*E*", NL="*NL*"):
    """
    Reconstructs plain text from a token list that includes START/END/NL markers.
    ("reverts" text_to_list)
    Args:
        text_list: List of tokens produced by `text_to_list` or the generator.
        separator: Separator to use between logical chunks when END markers are hit.
        avoid_double_newlines: If True, collapse consecutive newline markers to a single newline.
        START: Token to mark the start of a chunk.
        END: Token to mark the end of a chunk.
        NL: Placeholder for newlines.
    Returns:
        A cleaned text string without START/END indicators and NL converted into newlines.
    """

    out = []
    at_line_start = True

    for word in text_list:
        if word == START:
            continue
        if word == END:
            if not at_line_start and avoid_double_newlines:
                out.append("\n")
            at_line_start = True
        elif word == NL:
            if not at_line_start and avoid_double_newlines:
                out.append("\n")
            at_line_start = True
        else:
            if not at_line_start:
                out.append(" ")
            out.append(word)
            at_line_start = False

    return "".join(out).strip()

## Dictionary building functions

In [5]:
def build_markov_model(markov_model, new_text, order=1, separator="\n\n", START="*S*", END="*E*", NL="*NL*"):
    """
    Update a count dictionary that defines a markov model using new text.

    Args:
        markov_model: Dict mapping state tuples to next-token count dicts.
        new_text: Training text.
        order: Markov order (token length of the count dictionary's key).
        separator: Chunk separator passed to `text_to_list`.
        START: Token to mark the start of a chunk.
        END: Token to mark the end of a chunk.
        NL: Placeholder for newlines.

    Returns:
        The same markov_model dict updated with transition counts learned from input text.
    """

    text_list = text_to_list(new_text, separator=separator)

    expanded = []
    for word in text_list:
        if word == START:
            expanded.extend([START] * order)
        else:
            expanded.append(word)

    for i in range(len(expanded) - order):
        state = tuple(expanded[i : i + order])
        nxt = expanded[i + order]

        if state not in markov_model:
            markov_model[state] = {}
        if nxt not in markov_model[state]:
            markov_model[state][nxt] = 0

        markov_model[state][nxt] += 1

    return markov_model

In [6]:
def normalize_model(markov_model_counts):
    """
    Convert a count-based Markov model into per-state probability distributions.
    Args:
        markov_model_counts: Dict of {state: {next_token: count}}, produced by `build_markov_model`.
    Returns:
        Dict of {state: {next_token: probability}} where probabilities in each state have a sum = 1.
    """

    normalized = {}

    for k, v in markov_model_counts.items():
        # k is a tuple, v is a dict, vv is an int

        total = sum(v.values())
        if total == 0:
            continue

        normalized[k] = {}
        for v, vv in v.items():
            normalized[k][v] = vv / total

    return normalized

## Generation functions

In [7]:
def get_next_word(current_state, markov_model_normalized, rng_object, END="*E*"):
    """
    Sample the next token for a state using weighted random choice.
    Args:
        current_state: Tuple of tokens ("words") representing the current Markov state.
        markov_model_normalized: {state: {next_token: probability}} dict from `normalize_model`.
        rng_object: random.Random instance used for reproducible random sampling.
        END: Token to mark the end of a chunk.
    Returns:
        The sampled next token, or END if the state is unrecognized.
    """
    dist_dict = markov_model_normalized.get(current_state)
    if not dist_dict:
        # end it if cant be found
        return END

    words = list(dist_dict.keys())
    probs = list(dist_dict.values())

    return rng_object.choices(words, weights=probs, k=1)[0]

# another method for get_next_word
def get_next_word_alternative(current_state, markov_model_normalized, rng_object, END="*E*"):
    """
    Alternate sampler that draws the next token via cumulative probability.
    Args:
        current_state: Tuple of tokens ("words") representing the current Markov state.
        markov_model_normalized: {state: {next_token: probability}} dict from `normalize_model`.
        rng_object: random.Random instance used for reproducible random sampling.
        END: Token to mark the end of a chunk.
    Returns:
        The sampled next token, or END if the state is unrecognized.
    """

    dist_dict = markov_model_normalized.get(current_state)
    if not dist_dict:
        # end it if cant be found
        return END

    r = rng_object.random()
    cumulative = 0.0

    for next_word in dist_dict .keys():
      cumulative += dist_dict [next_word]
      if r <= cumulative:
        return next_word

    return next_word

In [8]:
def generate_random_text(markov_model_normalized, seed, order=1,
    limit_by="word", limit=200, separator="\n\n",
    hard_limit=2000, 
    START="*S*", END="*E*", NL="*NL*"):
    """
    Generate text from a normalized Markov model with configurable stopping rules.
    Args:
        markov_model_normalized: {state: {next_token: probability}} dict used for sampling.
        seed: Seed for the internal RNG to make output deterministic.
        order: Markov order (length of state). Default 1.
        limit_by: Stop criterion: one of {"word", "line", "probabilistic"}. Default "word".
            "word": generation continues until (limit) number of tokens ("words") have been added
            "line": geneeration continues until (limit) number of newlines have been added
        limit: Maximum words/lines, used by limit_by. Default 200.
        separator: Chunk separator passed through to `list_to_text`. Default "\n\n"
        hard_limit: Absolute cap on generated tokens to prevent runaway loops. Default 2000.
        START: Token to mark the start of a chunk.
        END: Token to mark the end of a chunk.
        NL: Placeholder for newlines.
    Returns:
        A string of generated text.
    """

    if order < 1:
        raise ValueError("order must be >= 1")
    if limit_by not in ("word", "line", "probabilistic"):
        raise ValueError('limit_by must be "word", "line",or "probabilistic"')
    if limit < 1:
        raise ValueError("limit must be >= 1")
    if hard_limit < 1:
        raise ValueError("hard_limit must be >= 1")

    rng_object = random.Random(seed)

    generated_list = [START] * order
    word_count = 0
    line_count = 0

    while True:
        if len(generated_list) >= hard_limit:
            break
        if limit_by == "word" and word_count >= limit:
            break
        if limit_by == "line" and line_count >= limit:
            break

        state = tuple(generated_list[-order:])
        nxt = get_next_word(state, markov_model_normalized, rng_object)
        
        if limit_by == "probabilistic" and nxt == END:
            break

        generated_list.append(nxt)

        if nxt == NL:
            line_count += 1
        elif nxt not in (START, END):
            word_count += 1

    return list_to_text(generated_list, separator=separator)

# Example implementation

## Example: 14 line sonnet
Using the limit_by and limit parameters, we can specify how many newlines must occur before stopping, unless the hard cap is reached.

In [9]:
# set parameters
path = "data/sonnets.txt"
START = "*S*"
END = "*E*"
NL = "*NL*"
seed = 543212345
order = 2
limit_by = "line"
limit = 14
separator = "\n\n"

# read new text from file
new_text = read_text_file(path)

# initialize and update the dictionary
markov_model = {}
build_markov_model(markov_model, new_text, order=order, separator=separator, START=START, END=END, NL=NL)

# create normalized dictionary
# note that the currently used method for get_next_word does not strictly require normalization 
markov_model_normalized = normalize_model(markov_model)

# generate
output = generate_random_text(
    markov_model_normalized,
    seed = seed,
    order = order,
    limit_by = limit_by,
    limit = limit,
    separator = separator,
    START=START, END=END, NL=NL
    )

#print
print(output)


like as, to prevent our maladies unseen,
we sicken to shun sickness when we purge;
even for this, let us divided live,
and found it in thy cheek: he can tell
that heals the wound, and cures not the disgrace:
nor dare i question with my extern the outward honouring,
or ten times more in worth
than of your fame!
but that which thou deserv'st alone.
and in fresh numbers number all your graces,
and though they themselves be bevel;
by new unfolding his imprison'd pride.
blessed are you made,
that in guess they measure by thy dial's shady stealth mayst know


## Example: probabilistic stopping
When stopping is probabilistic, the generation has a chance to END if the current state is at least once followed by a sonnet end in the original text. The shortest possible output is thus a single word long, and the longest possible output hits the hard limit.

With the same seed as the above, changing the limit parameter to "probabilistic" results in one extra line. 

In [10]:
# set parameters
path = "data/sonnets.txt"
START = "*S*"
END = "*E*"
NL = "*NL*"
seed = 543212345
order = 2
limit_by = "probabilistic"
limit = 14
separator = "\n\n"

# read new text from file
new_text = read_text_file(path)

# initialize and update the dictionary
markov_model = {}
build_markov_model(markov_model, new_text, order=order, separator=separator, START=START, END=END, NL=NL)

# create normalized dictionary
# note that the currently used method for get_next_word does not strictly require normalization 
markov_model_normalized = normalize_model(markov_model)

# generate
output = generate_random_text(
    markov_model_normalized,
    seed = seed,
    order = order,
    limit_by = limit_by,
    limit = limit,
    separator = separator,
    START=START, END=END, NL=NL
    )

#print
print(output)

like as, to prevent our maladies unseen,
we sicken to shun sickness when we purge;
even for this, let us divided live,
and found it in thy cheek: he can tell
that heals the wound, and cures not the disgrace:
nor dare i question with my extern the outward honouring,
or ten times more in worth
than of your fame!
but that which thou deserv'st alone.
and in fresh numbers number all your graces,
and though they themselves be bevel;
by new unfolding his imprison'd pride.
blessed are you made,
that in guess they measure by thy dial's shady stealth mayst know
that i may not be foes.
